# Data audit: January 2024

Inspect missing values, pickup-zone coverage, and January records before hourly aggregation.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq


In [ ]:
path = str(Path.home() / "Downloads" / "yellow_tripdata_2024-01.parquet")
table = pq.read_table(path)
df = table.to_pandas()
zones = pd.read_csv(str(Path.home() / "Downloads" / "taxi_zone_lookup.csv"))


In [ ]:
df.head()


In [ ]:
zones.head()


#### Missing values


Zone IDs 264 and 265 have no usable zone information. Check their share of recorded pickups before excluding them.


In [ ]:
Location_id_miss1 = 264
Location_id_miss2 = 265
m1 = df.PULocationID[df.PULocationID == Location_id_miss1].count()
m2 = df.PULocationID[df.PULocationID == Location_id_miss2].count()
mtot = len(df)
print(m1/mtot, m2/mtot)


#### Zones 264 and 265 account for approximately 0.4% of pickups


In [ ]:
time1 = df.set_index("tpep_pickup_datetime").between_time("8:00", "11:00", inclusive = "left")
time2 = df.set_index("tpep_pickup_datetime").between_time("11:00", "14:00", inclusive = "left")
time3 = df.set_index("tpep_pickup_datetime").between_time("14:00", "17:00", inclusive = "left")
time4 = df.set_index("tpep_pickup_datetime").between_time("17:00", "20:00", inclusive = "left")
time5 = df.set_index("tpep_pickup_datetime").between_time("20:00", "23:00", inclusive = "left")
time6 = df.set_index("tpep_pickup_datetime").between_time("23:00", "2:00", inclusive = "left")
time7 = df.set_index("tpep_pickup_datetime").between_time("2:00", "5:00", inclusive = "left")
time8 = df.set_index("tpep_pickup_datetime").between_time("5:00", "8:00", inclusive = "left")
times = [time1, time2, time3, time4, time5, time6, time7, time8]


In [ ]:
#264 and 265
L264 = []
L265 = []
for i in range(len(times)):
    L264.append(times[i].PULocationID[times[i].PULocationID == 264].count()/len(times[i]))
    L265.append(times[i].PULocationID[times[i].PULocationID == 265].count()/len(times[i]))
print(L264, L265)


The share varies slightly by time of day but remains small. Exclude pickups assigned to these two zone IDs.


In [ ]:
df_clean = df.copy()

def clean(df):
    return df[(df["PULocationID"] != 265) & (df["PULocationID"] != 264)]

df_clean = clean(df_clean)


#### Overlap of the missing values


In [ ]:
cols = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "Airport_fee",
    "congestion_surcharge"
]

missing = df_clean[cols].isna().astype(int)

overlap = missing.T @ missing
overlap


#### Checking for invalid values


In [ ]:
df_clean.tpep_dropoff_datetime.dtype


#### Select pickups within the calendar month


In [ ]:
def valid_date(df, m):
    start = pd.Timestamp(year = 2024, month = m, day = 1)
    end = start + pd.DateOffset(months = 1)
    df_clean = df.copy()
    mask1 = (pd.to_datetime(df_clean["tpep_pickup_datetime"]) >= start)
    mask2 = (pd.to_datetime(df_clean["tpep_pickup_datetime"]) < end)
    df_clean = df_clean[mask1]
    df_clean = df_clean[mask2]
    return df_clean


In [ ]:
df_clean.to_parquet(
    str(project_root / "data/yellow_taxi_2024_01_cleaned.parquet"),
    index = False
)
